# ShuttleSet Frame Extraction Pipeline (NOT USED YET)

Extracts **all frames between consecutive strokes** within each rally from ShuttleSet match videos.

**Process:**
1. Load match metadata & annotations (all sets per match)
2. Download video from YouTube via yt-dlp
3. Read actual video FPS to verify frame_num ↔ time alignment
4. Extract all frames between consecutive stroke frame_nums within each rally
5. Save structured JSON with frame paths, time, and annotation features
6. Verify extraction visually (chronological order, CJK labels)

In [2]:
!pip install matplotlib inline

  Using cached matplotlib-3.10.8-cp310-cp310-macosx_11_0_arm64.whl.metadata (52 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached contourpy-1.3.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp310-cp310-macosx_10_9_universal2.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp310-cp310-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.2-cp310-cp310-macosx_11_0_arm64.whl (253 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.61.1-cp310-cp310-macosx_10_9_universal2.whl (2.9 MB)
Using cached kiwisolver-1.4.9-cp310-cp310-macosx_11_0_arm64.whl (65 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl 

In [3]:
import subprocess
import os
import json
import csv
from pathlib import Path

import pandas as pd
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

## 1. Config

In [4]:
NOTEBOOK_DIR = Path(".").resolve()
TMP_VIDEO = NOTEBOOK_DIR / "tmp_video.mp4"
SHUTTLESET_ROOT = NOTEBOOK_DIR / "ShuttleSet"
MATCHES_CSV = SHUTTLESET_ROOT / "set" / "match.csv"
ANNOT_ROOT = SHUTTLESET_ROOT / "set"
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs"
FRAMES_ROOT = NOTEBOOK_DIR / "frames"

OUTPUT_ROOT.mkdir(exist_ok=True)
FRAMES_ROOT.mkdir(exist_ok=True)

SHOT_TYPES = [
    '發短球', '發長球', '推撲球', '殺球', '過渡球', '防守回挑',
    '切球', '接殺防守', '長球', '平球', '擋小球', '挑球',
    '放小球', '勾球', '網前球', '點扣', '推球', '未知'
]

print(f"Annotations root: {ANNOT_ROOT}")
print(f"Match CSV: {MATCHES_CSV} (exists: {MATCHES_CSV.exists()})")
print(f"Output dir: {OUTPUT_ROOT}")
print(f"Frames dir: {FRAMES_ROOT}")

Annotations root: /Users/yuen@backbase.com/Documents/Baddiev2/Datasets/ShuttleSet/set
Match CSV: /Users/yuen@backbase.com/Documents/Baddiev2/Datasets/ShuttleSet/set/match.csv (exists: True)
Output dir: /Users/yuen@backbase.com/Documents/Baddiev2/Datasets/outputs
Frames dir: /Users/yuen@backbase.com/Documents/Baddiev2/Datasets/frames


## 2. Load Match Metadata

In [5]:
match_df = pd.read_csv(MATCHES_CSV)
match_df.columns = match_df.columns.str.strip()
for col in match_df.select_dtypes(include='object').columns:
    match_df[col] = match_df[col].str.strip()

print(f"Total matches in dataset: {len(match_df)}")
match_df[["id", "video", "tournament", "winner", "loser", "set"]].head(10)

Total matches in dataset: 44


,id,video,tournament,winner,loser,set
0,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,Fuzhou Open 2019,Kento MOMOTA,CHOU Tien Chen,3
1,2,CHEN_Long_CHOU_Tien_Chen_World_Tour_Finals_Gro...,World Tour Finals,CHEN Long,CHOU Tien Chen,2
2,3,Kento_MOMOTA_CHOU_Tien_Chen_KOREA_OPEN_2019_Final,KOREA OPEN 2019,Kento MOMOTA,CHOU Tien Chen,2
3,4,CHEN_Long_CHOU_Tien_Chen_Denmark_Open_2019_Qua...,Denmark Open 2019,CHEN Long,CHOU Tien Chen,2
4,5,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2018_F...,Fuzhou Open 2018,Kento MOMOTA,CHOU Tien Chen,3
5,6,Kento_MOMOTA_CHOU_Tien_Chen_Denmark_Open_2018_...,Denmark Open 2018,Kento MOMOTA,CHOU Tien Chen,3
6,7,Kento_MOMOTA_CHOU_Tien_Chen_Malaysia_Open_2018...,Malaysia Open 2018,Kento MOMOTA,CHOU Tien Chen,2
7,8,CHOU_Tien_Chen_Anders_ANTONSEN_Fuzhou_Open_201...,Fuzhou Open 2019,CHOU Tien Chen,Anders ANTONSEN,2
8,9,CHOU_Tien_Chen_Jonatan_CHRISTIE_Sudirman_Cup_2...,Sudirman Cup 2019,CHOU Tien Chen,Jonatan CHRISTIE,2
9,10,CHOU_Tien_Chen_NG_Ka_Long_Angus_Sudirman_Cup_2...,Sudirman Cup 2019,CHOU Tien Chen,NG Ka Long Angus,2


In [6]:
# Select which matches to process (adjust slice to control batch size)
matches_to_process = match_df  # all matches; use match_df.head(5) to limit
print(f"Will process {len(matches_to_process)} matches")

Will process 44 matches


## 3. Core Functions

In [7]:
def parse_time_to_seconds(time_str):
    """Parse CSV time column (e.g. '0:07:39' or '00:07:43') to total seconds"""
    if pd.isna(time_str) or not str(time_str).strip():
        return None
    parts = str(time_str).strip().split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    elif len(parts) == 2:
        return int(parts[0]) * 60 + int(parts[1])
    return None


def download_video(url: str):
    """Download one video to a temp file"""
    if TMP_VIDEO.exists():
        os.remove(TMP_VIDEO)
    print("[INFO] Downloading video...")
    subprocess.run([
        "yt-dlp",
        "-f", "bv*[ext=mp4]/bv*",
        "-o", str(TMP_VIDEO),
        url
    ], check=True)


def load_all_set_annotations(match_dir: Path):
    """Load annotations from ALL sets (set1.csv, set2.csv, set3.csv, ...)"""
    all_dfs = []
    for set_file in sorted(match_dir.glob("set*.csv")):
        df = pd.read_csv(set_file)
        df["_set_file"] = set_file.name
        all_dfs.append(df)
        print(f"  Loaded {set_file.name}: {len(df)} strokes")

    if not all_dfs:
        return None

    combined = pd.concat(all_dfs, ignore_index=True)
    combined = combined[
        combined["frame_num"].notna() &
        combined["type"].notna() &
        (combined["type"] != "")
    ].copy()
    combined["frame_num"] = combined["frame_num"].astype(int)
    combined["time_seconds"] = combined["time"].apply(parse_time_to_seconds)
    return combined


def collect_rally_frame_ranges(annotations_df):
    """Collect all frame ranges between consecutive strokes within each rally.
    
    Returns:
        set of all frame numbers to extract (every frame between consecutive strokes)
    """
    all_frames = set()
    
    # Group by set file and rally
    for (set_file, rally), group in annotations_df.groupby(["_set_file", "rally"]):
        group_sorted = group.sort_values("frame_num")
        frame_nums = group_sorted["frame_num"].values
        
        # Add all frames from first stroke to last stroke in this rally
        if len(frame_nums) >= 1:
            start = int(frame_nums[0])
            end = int(frame_nums[-1])
            for f in range(start, end + 1):
                all_frames.add(f)
    
    return sorted(f for f in all_frames if f >= 0)


def save_frame(frame, frame_path: Path):
    """Save frame locally as PNG"""
    frame_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(frame_path), frame)

In [8]:
def verify_fps_alignment(annotations_df, video_fps):
    """Check that CSV time column matches frame_num / video_fps.
    Prints first few comparisons so you can verify alignment."""
    print(f"\n  Video FPS: {video_fps:.2f}")
    print(f"  {'CSV Time':>12s}  {'CSV Secs':>8s}  {'frame_num':>10s}  {'frame/fps':>10s}  {'Delta':>6s}")
    print(f"  {'-'*12}  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*6}")
    
    sample = annotations_df.head(10)
    for _, row in sample.iterrows():
        csv_secs = row["time_seconds"]
        frame_secs = row["frame_num"] / video_fps if video_fps > 0 else 0
        delta = abs(csv_secs - frame_secs) if csv_secs is not None else None
        delta_str = f"{delta:.1f}s" if delta is not None else "N/A"
        print(f"  {str(row['time']):>12s}  {csv_secs:>8.0f}  {row['frame_num']:>10d}  {frame_secs:>10.1f}  {delta_str:>6s}")
    
    # Summary
    valid = annotations_df[annotations_df["time_seconds"].notna()]
    computed = valid["frame_num"] / video_fps
    deltas = (valid["time_seconds"] - computed).abs()
    print(f"\n  Alignment stats (all {len(valid)} strokes):")
    print(f"    Mean delta: {deltas.mean():.2f}s")
    print(f"    Max delta:  {deltas.max():.2f}s")
    if deltas.mean() > 2.0:
        print("    ⚠ WARNING: Large mean delta — FPS mismatch likely!")
    else:
        print("    ✓ Alignment looks good")

In [9]:
def process_match(video_url: str, match_id: str):
    print(f"\n=== Processing match: {match_id} ===")

    match_dir = ANNOT_ROOT / match_id
    if not match_dir.exists():
        print(f"[SKIP] Annotation dir not found: {match_dir}")
        return None

    # 1. Load ALL set annotations
    annotations_df = load_all_set_annotations(match_dir)
    if annotations_df is None or annotations_df.empty:
        print(f"[SKIP] No valid annotations for {match_id}")
        return None

    # Shot type distribution for this match
    type_counts = annotations_df["type"].value_counts()
    print(f"  Total strokes: {len(annotations_df)}, types: {len(type_counts)}")
    for shot, count in type_counts.head(5).items():
        print(f"    {shot}: {count}")

    # 2. Download video
    download_video(video_url)

    # 3. Read actual video FPS and verify alignment
    cap = cv2.VideoCapture(str(TMP_VIDEO))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"  Video: {total_video_frames} frames, {video_fps:.2f} fps")
    
    verify_fps_alignment(annotations_df, video_fps)

    # 4. Collect ALL frames between consecutive strokes within each rally
    all_frame_nums = collect_rally_frame_ranges(annotations_df)
    print(f"[INFO] Total frames to extract: {len(all_frame_nums)}")

    # 5. Extract and save frames (streaming, sequential seek)
    match_frames_dir = FRAMES_ROOT / match_id
    extracted = 0
    for fn in all_frame_nums:
        if fn < 0 or fn >= total_video_frames:
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, fn)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_path = match_frames_dir / f"frame_{fn:06d}.png"
        save_frame(frame, frame_path)
        extracted += 1

    cap.release()
    print(f"[INFO] Extracted {extracted}/{len(all_frame_nums)} frames")

    # 6. Build per-stroke records
    stroke_records = []
    sorted_annot = annotations_df.sort_values(["_set_file", "rally", "ball_round"]).reset_index(drop=True)
    
    for i, row in sorted_annot.iterrows():
        fn = int(row["frame_num"])
        
        # Determine frame range for this stroke:
        # from this stroke's frame_num to the next stroke's frame_num (exclusive)
        # Within the same rally only
        next_fn = None
        if i + 1 < len(sorted_annot):
            next_row = sorted_annot.iloc[i + 1]
            if (next_row["_set_file"] == row["_set_file"] and 
                next_row["rally"] == row["rally"]):
                next_fn = int(next_row["frame_num"])
        
        # Build list of all frame paths for this stroke's duration
        if next_fn is not None:
            stroke_frame_nums = list(range(fn, next_fn))
        else:
            # Last stroke in rally — just include the stroke frame
            stroke_frame_nums = [fn]
        
        frame_paths = [
            str(match_frames_dir / f"frame_{f:06d}.png")
            for f in stroke_frame_nums
        ]

        record = {
            "match_id": match_id,
            "set_file": row.get("_set_file", ""),
            "rally": int(row["rally"]) if pd.notna(row.get("rally")) else None,
            "ball_round": int(row["ball_round"]) if pd.notna(row.get("ball_round")) else None,
            "frame_num": fn,
            "frame_end": next_fn if next_fn else fn,
            "num_frames": len(stroke_frame_nums),
            "time": str(row["time"]) if pd.notna(row.get("time")) else None,
            "time_seconds": float(row["time_seconds"]) if pd.notna(row.get("time_seconds")) else None,
            "video_fps": video_fps,
            "type": row["type"],
            "player": row.get("player", ""),
            "frame_paths": frame_paths,
            # Court position features from annotations
            "hit_area": row.get("hit_area"),
            "hit_x": row.get("hit_x"),
            "hit_y": row.get("hit_y"),
            "landing_area": row.get("landing_area"),
            "landing_x": row.get("landing_x"),
            "landing_y": row.get("landing_y"),
            "player_location_x": row.get("player_location_x"),
            "player_location_y": row.get("player_location_y"),
            "opponent_location_x": row.get("opponent_location_x"),
            "opponent_location_y": row.get("opponent_location_y"),
            "backhand": row.get("backhand"),
            "hit_height": row.get("hit_height"),
        }
        stroke_records.append(record)

    # 7. Save structured output
    out_path = OUTPUT_ROOT / f"{match_id}.json"
    with open(out_path, "w") as f:
        json.dump(stroke_records, f, indent=2, ensure_ascii=False)

    # 8. Cleanup video
    if TMP_VIDEO.exists():
        os.remove(TMP_VIDEO)

    print(f"[DONE] {match_id}: {len(stroke_records)} stroke records saved")
    return stroke_records

## 4. Run Extraction

In [12]:
summary = {"total_matches": 0, "total_strokes": 0, "shot_distribution": {}}

for _, row in matches_to_process.iterrows():
    match_id = row["video"]
    url = row["url"]

    if pd.isna(url) or not url:
        print(f"[SKIP] No URL for {match_id}")
        continue

    # Skip already-processed matches
    out_file = OUTPUT_ROOT / f"{match_id}.json"
    if out_file.exists():
        print(f"[SKIP] Already processed: {match_id}")
        with open(out_file) as jf:
            records = json.load(jf)
    else:
        try:
            # Ensure yt-dlp is installed and up-to-date
            import yt_dlp
        except ImportError:
            # Install yt-dlp if not present
            get_ipython().run_line_magic('pip', 'install -U yt-dlp')
            import yt_dlp

        try:
            records = process_match(url, match_id)
        except Exception as e:
            print(f"[ERROR] Failed to process match {match_id}: {e}")
            continue
        if records is None:
            continue

    summary["total_matches"] += 1
    summary["total_strokes"] += len(records)
    for r in records:
        shot = r.get("type", "未知球種")
        summary["shot_distribution"][shot] = summary["shot_distribution"].get(shot, 0) + 1

# Save summary
with open(OUTPUT_ROOT / "pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n=== ALL MATCHES PROCESSED ===")
print(f"  Matches: {summary['total_matches']}")
print(f"  Total strokes: {summary['total_strokes']}")
print(f"  Shot types: {len(summary['shot_distribution'])}")

[SKIP] Already processed: Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals
[SKIP] Already processed: CHEN_Long_CHOU_Tien_Chen_World_Tour_Finals_Group_Stage
[SKIP] Already processed: Kento_MOMOTA_CHOU_Tien_Chen_KOREA_OPEN_2019_Final
[SKIP] Already processed: CHEN_Long_CHOU_Tien_Chen_Denmark_Open_2019_QuarterFinal
[SKIP] Already processed: Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2018_Finals
[SKIP] Already processed: Kento_MOMOTA_CHOU_Tien_Chen_Denmark_Open_2018_Finals
[SKIP] Already processed: Kento_MOMOTA_CHOU_Tien_Chen_Malaysia_Open_2018_QuarterFinals
[SKIP] Already processed: CHOU_Tien_Chen_Anders_ANTONSEN_Fuzhou_Open_2019_Semi-finals
[SKIP] Already processed: CHOU_Tien_Chen_Jonatan_CHRISTIE_Sudirman_Cup_2019_Quarter-finals
[SKIP] Already processed: CHOU_Tien_Chen_NG_Ka_Long_Angus_Sudirman_Cup_2019_Group_Stage
[SKIP] Already processed: CHOU_Tien_Chen_Jonatan_CHRISTIE_Indonesia_Open_2019_Quarter-finals
[SKIP] No URL for NG_Ka_Long_Angus_SHI_Yu_Qi_Thailand_Masters_2020_SemiFinals
[SK

[youtube] HTBf9wFL0mk: Downloading android vr player API JSON


ERROR: [youtube] HTBf9wFL0mk: Video unavailable. This video is not available


CalledProcessError: Command '['yt-dlp', '-f', 'bv*[ext=mp4]/bv*', '-o', '/Users/yuen@backbase.com/Documents/Baddiev2/Datasets/tmp_video.mp4', 'https://www.youtube.com/watch?v=HTBf9wFL0mk']' returned non-zero exit status 1.

## 5. Verify Extraction

Shows the first N strokes in chronological order. For each stroke, displays the first frame,
a mid-point frame, and the last frame of the stroke duration — with CJK shot labels and
the exact CSV timestamp.

In [ ]:
def get_cjk_font(size=24):
    """Find a system font that supports Chinese characters"""
    font_paths = [
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/Hiragino Sans GB.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    ]
    for fp in font_paths:
        if Path(fp).exists():
            return ImageFont.truetype(fp, size)
    return ImageFont.load_default()

FONT_LARGE = get_cjk_font(32)
FONT_MEDIUM = get_cjk_font(24)


def annotate_frame(frame_bgr, record, frame_label=""):
    """Draw annotation overlay using PIL for CJK support. Works on full-res frame."""
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    h, w = frame_bgr.shape[:2]

    # Line 1: set | rally | round | CSV time
    set_name = record.get('set_file', '').replace('.csv', '')
    time_str = record.get('time', '?')
    line1 = f"{set_name} | Rally {record['rally']} Round {record['ball_round']} | {time_str}"
    draw.text((10, 10), line1, font=FONT_MEDIUM, fill=(255, 255, 255))

    # Line 2: shot type | player | frame count
    line2 = f"{record['type']} | Player {record['player']} | {record.get('num_frames', '?')} frames"
    draw.text((10, 42), line2, font=FONT_LARGE, fill=(0, 255, 0))

    # Frame label (START / MID / END)
    if frame_label:
        draw.text((w - 120, 10), frame_label, font=FONT_MEDIUM, fill=(180, 180, 180))

    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)


def verify_match(match_id, num_samples=6):
    """Display first N strokes chronologically with start/mid/end frames"""
    json_path = OUTPUT_ROOT / f"{match_id}.json"
    if not json_path.exists():
        print(f"[ERROR] No output found: {json_path}")
        return

    with open(json_path) as f:
        records = json.load(f)

    # Chronological order
    sorted_records = sorted(records, key=lambda r: r["frame_num"])
    samples = sorted_records[:num_samples]

    print(f"Verifying: {match_id}")
    print(f"  Total strokes: {len(records)}, showing first {len(samples)}")
    print()
    for s in samples:
        print(f"  {s['set_file']:>8s} | Rally {s['rally']:>2} Round {s['ball_round']:>2} | "
              f"{s['time']:>10s} | frame {s['frame_num']:>6d} | {s['type']} | "
              f"Player {s['player']} | {s['num_frames']} frames")

    # Display: 3 columns (start, mid, end of stroke duration) x N rows
    fig, axes = plt.subplots(len(samples), 3, figsize=(20, 5 * len(samples)))
    if len(samples) == 1:
        axes = [axes]

    for row_idx, record in enumerate(samples):
        frame_paths = record.get("frame_paths", [])
        n = len(frame_paths)

        # Pick start, mid, end frames
        if n >= 3:
            indices = [0, n // 2, n - 1]
        elif n == 2:
            indices = [0, 0, 1]
        else:
            indices = [0, 0, 0]

        col_labels = ["START", "MID", "END"]

        for col_idx, (frame_idx, col_label) in enumerate(zip(indices, col_labels)):
            ax = axes[row_idx][col_idx]

            if frame_idx < len(frame_paths):
                fpath = Path(frame_paths[frame_idx])
                if fpath.exists():
                    img = cv2.imread(str(fpath))
                    img = annotate_frame(img, record, frame_label=col_label)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    ax.imshow(img)
                else:
                    ax.text(0.5, 0.5, f"MISSING\n{fpath.name}",
                            ha='center', va='center', transform=ax.transAxes,
                            color='red', fontsize=12)
            else:
                ax.text(0.5, 0.5, "N/A", ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=14)

            ax.set_title(f"{col_label} (frame {frame_idx + 1}/{n})", fontsize=10)
            ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# List all processed matches
processed = [p.stem for p in sorted(OUTPUT_ROOT.glob("*.json")) if p.stem != "pipeline_summary"]
print(f"Processed matches available: {len(processed)}")
for i, m in enumerate(processed):
    print(f"  {i}: {m}")

In [ ]:
# Verify a match (change index or set match_id directly)
if processed:
    verify_match(processed[0])

## 6. Summary Stats

In [ ]:
# Load or recompute summary
summary_path = OUTPUT_ROOT / "pipeline_summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)

print(f"Matches processed: {summary['total_matches']}")
print(f"Total strokes: {summary['total_strokes']}")
print(f"Shot types: {len(summary['shot_distribution'])}")

# Plot shot distribution
shots = sorted(summary["shot_distribution"].items(), key=lambda x: x[1], reverse=True)
labels, counts = zip(*shots)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(range(len(labels)), counts)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.set_title("Shot Type Distribution (All Matches)")
for i, c in enumerate(counts):
    ax.text(c + 5, i, str(c), va='center', fontsize=9)
plt.tight_layout()
plt.show()